In [34]:
import pandas as pd

In [35]:
url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-10.parquet'

In [36]:
df = pd.read_parquet(
    url,
    columns=[
        "lpep_pickup_datetime",
        "lpep_dropoff_datetime",
        "PULocationID",
        "DOLocationID",
        "passenger_count",
        "trip_distance",
        "tip_amount",
        "total_amount",
    ],
)

In [37]:
df.head()

,lpep_pickup_datetime,lpep_dropoff_datetime,PULocationID,DOLocationID,passenger_count,trip_distance,tip_amount,total_amount
0,2025-10-01 00:21:47,2025-10-01 00:24:37,247,69,1.0,0.70,1.70,10.00
1,2025-10-01 00:14:03,2025-10-01 00:24:14,66,25,1.0,1.61,2.78,16.68
2,2025-10-01 00:16:44,2025-10-01 00:16:47,244,244,1.0,0.00,2.20,13.20
3,2025-10-01 00:07:36,2025-10-01 00:32:14,95,170,1.0,10.37,11.31,67.85
4,2025-09-30 21:10:29,2025-09-30 21:22:30,82,138,1.0,4.07,6.82,34.12


In [38]:
df.iloc[0]

lpep_pickup_datetime     2025-10-01 00:21:47
lpep_dropoff_datetime    2025-10-01 00:24:37
PULocationID                             247
DOLocationID                              69
passenger_count                          1.0
trip_distance                            0.7
tip_amount                               1.7
total_amount                            10.0
Name: 0, dtype: object

In [46]:
from dataclasses import dataclass


@dataclass
class Ride:
    PULocationID: int
    DOLocationID: int
    trip_distance: float
    total_amount: float
    lpep_pickup_datetime: int  # epoch milliseconds

# Example
ride = Ride(
    PULocationID=1,
    DOLocationID=10,
    trip_distance=10,
    total_amount=20,
    lpep_pickup_datetime=1761956005000,
)

In [48]:
# NB: Kafka works with raw bytes, so we need a serializer that converts Python dicts to JSON
# NB2: In production, we would have several bootstrap_servers for redundancy - if one is down, the client connects through another.
import json
from kafka import KafkaProducer


def json_serializer(data):
    return json.dumps(data).encode("utf-8")


server = "localhost:9092"

producer = KafkaProducer(bootstrap_servers=[server], value_serializer=json_serializer)

In [49]:
import dataclasses

topic_name = "rides"
producer.send(topic_name, value=dataclasses.asdict(ride))
producer.flush()

In [ ]:
import time
import dataclasses
from models import ride_from_row, ride_serializer


server = "localhost:9092"
producer = KafkaProducer(bootstrap_servers=[server], value_serializer=ride_serializer)
producer.send(topic_name, value=ride)
producer.flush()

t0 = time.time()

for _, row in df.iterrows():
    ride = ride_from_row(row)
    producer.send(topic_name, value=ride)
    print(f"Sent: {ride}")
    time.sleep(0.01)

producer.flush()

t1 = time.time()
print(f"took {(t1 - t0):.2f} seconds")

Sent: Ride(PULocationID=247, DOLocationID=69, trip_distance=0.7, tip_amount=1.7, passenger_count=1, total_amount=10.0, lpep_pickup_datetime=1759278107000, lpep_dropoff_datetime=1759278277000)
Sent: Ride(PULocationID=66, DOLocationID=25, trip_distance=1.61, tip_amount=2.78, passenger_count=1, total_amount=16.68, lpep_pickup_datetime=1759277643000, lpep_dropoff_datetime=1759278254000)
Sent: Ride(PULocationID=244, DOLocationID=244, trip_distance=0.0, tip_amount=2.2, passenger_count=1, total_amount=13.2, lpep_pickup_datetime=1759277804000, lpep_dropoff_datetime=1759277807000)
Sent: Ride(PULocationID=95, DOLocationID=170, trip_distance=10.37, tip_amount=11.31, passenger_count=1, total_amount=67.85, lpep_pickup_datetime=1759277256000, lpep_dropoff_datetime=1759278734000)
Sent: Ride(PULocationID=82, DOLocationID=138, trip_distance=4.07, tip_amount=6.82, passenger_count=1, total_amount=34.12, lpep_pickup_datetime=1759266629000, lpep_dropoff_datetime=1759267350000)
Sent: Ride(PULocationID=129, 